In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Helper function
def load_log(filename):

    df = pd.read_csv(filename, header=None)

    df.columns = [
        "frame_id",
        "address",
        "retcode",
        "numDistances",
        "raw_mm",
        "medianEMA_mm",
        "medianEMA_in",
        "p0_str",
        "time_now",
        "status"
    ]

    df["time_now"] = pd.to_numeric(df["time_now"], errors="coerce")
    df["raw_mm"] = pd.to_numeric(df["raw_mm"], errors="coerce")
    df["medianEMA_mm"] = pd.to_numeric(df["medianEMA_mm"], errors="coerce")

    # Replace invalid measurements
    df["raw_mm"] = df["raw_mm"].replace(-1, np.nan)
    df["medianEMA_mm"] = df["medianEMA_mm"].replace(-1, np.nan)

    # Convert to inches
    df["raw_in"] = df["raw_mm"] / 25.4
    df["medianEMA_in"] = df["medianEMA_mm"] / 25.4

    return df


results = []

# Find all files under dataset/
all_files = sorted(Path("dataset").rglob("*"))

for file in all_files:

    if not file.is_file():
        continue

    # Example filename:
    # stationary_12_1.txt

    parts = file.stem.split("_")

    if len(parts) != 3:
        continue

    test_type = parts[0]

    if test_type != "stationary":
        continue

    target_height = int(parts[1])
    trial = int(parts[2])

    bed = file.parent.name

    df = load_log(file)

    results.append({
        "bed": bed,
        "height_in": target_height,
        "trial": trial,

        "samples": len(df)-1,

        "raw_mean_in":
            df["raw_in"].mean(),

        "raw_std_in":
            df["raw_in"].std(),

        "filtered_mean_in":
            df["medianEMA_in"].mean(),

        "filtered_std_in":
            df["medianEMA_in"].std(),

        "error_in":
            df["medianEMA_in"].mean() - target_height,

        "percent_invalid":
            100 * df["raw_mm"].isna().mean()
    })

summary = pd.DataFrame(results)

# Sort nicely
summary = summary.sort_values(
    ["bed", "height_in", "trial"]
)

print(summary)

# Optional: save to CSV
summary.to_csv(
    "stationary_summary.csv",
    index=False
)

     bed  height_in  trial  samples  raw_mean_in  raw_std_in  \
0   bed2         12      1      174    12.182324    0.147025   
1   bed2         12      2      167    13.254116    4.632530   
2   bed2         12      3      170    12.893932    0.079001   
3   bed2         16      1      165    16.118926    0.172135   
4   bed2         16      2      338    16.180127    1.364380   
5   bed2         16      3      174    16.774058    2.998979   
6   bed2         20      1      164    21.203565    1.644905   
7   bed2         20      2      174    20.110870    0.284723   
8   bed2         20      3      158    20.065285    0.301697   
9   bed3         12      1      169    11.615640    0.329537   
10  bed3         12      2      170    11.733905    0.048523   
11  bed3         12      3      171    11.353778    0.066329   
12  bed3         16      1      163    17.144099    0.309668   
13  bed3         16      2      199    16.547168    4.123486   
14  bed3         16      3      173    1